In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import sklearn
import math
import statistics
import warnings
warnings.filterwarnings(action='ignore')
pd.options.display.float_format = '{:.5f}'.format
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

temp_data=pd.read_csv("/Users/kbh/Downloads/lending_club_2020_train.csv",low_memory=False)

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)

In [ ]:
temp_data.drop(956843,axis=0,inplace=True)

In [ ]:
temp_data.shape

In [ ]:
temp_data['term']=temp_data['term'].str[:3].astype(int)
temp_data['int_rate']=temp_data['int_rate'].str[:6].astype(float)
temp_data=temp_data.replace({'emp_length':'< 1 year'},'0.5',)
temp_data['emp_length']=temp_data.emp_length.str.extract(r'(\d\d|\d+.\d+|\d)').astype(float)
temp_data['last_pymnt_d'] = pd.to_datetime(temp_data['last_pymnt_d'], format='%b-%Y').dt.to_period('M')
temp_data['last_credit_pull_d'] = pd.to_datetime(temp_data['last_credit_pull_d'], format='%b-%Y').dt.to_period('M')

?? ???

In [ ]:
labelencoding=LabelEncoder()
temp_data.sub_grade=labelencoding.fit_transform(temp_data['sub_grade'].astype(str))

NA ?? ?? ??? ??(??)

In [ ]:
eliminate_list='''annual_inc
delinq_2yrs
earliest_cr_line
inq_last_6mths
open_acc
pub_rec
total_acc
acc_now_delinq
tot_coll_amt
tot_cur_bal
delinq_amnt
mo_sin_old_rev_tl_op
mo_sin_rcnt_rev_tl_op
mo_sin_rcnt_tl
num_accts_ever_120_pd
num_actv_bc_tl
num_actv_rev_tl
num_bc_tl
num_il_tl
num_op_rev_tl
num_rev_accts
num_rev_tl_bal_gt_0
num_tl_30dpd
num_tl_90g_dpd_24m
num_tl_op_past_12m
tot_hi_cred_lim
total_il_high_credit_limit
last_credit_pull_d
'''
## last_credit_pull_d << ???
eliminate_list = eliminate_list.strip().split('\n')


NA? 0?? ??

In [ ]:
replace_zero_list='''mths_since_last_delinq
annual_inc_joint
dti_joint
verification_status_joint
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_cu_tl
inq_last_12m
acc_open_past_24mths
revol_bal_joint
sec_app_fico_range_low
sec_app_fico_range_high
sec_app_earliest_cr_line
sec_app_inq_last_6mths
sec_app_mort_acc
sec_app_open_acc
sec_app_revol_util
sec_app_open_act_il
sec_app_num_rev_accts
sec_app_chargeoff_within_12_mths
sec_app_collections_12_mths_ex_med
inq_fi
emp_length
'''
replace_zero_list = replace_zero_list.strip().split('\n')

? ??

In [ ]:
eliminate_col='''emp_title
url
title
zip_code
addr_state
mths_since_last_record
next_pymnt_d
hardship_flag
hardship_type
hardship_reason
hardship_status
deferral_term
hardship_amount
hardship_start_date
hardship_end_date
payment_plan_start_date
hardship_length
hardship_dpd
hardship_loan_status
hardship_payoff_balance_amount
hardship_last_payment_amount
orig_projected_additional_accrued_interest
grade'''
eliminate_col = eliminate_col.strip().split('\n')

temp_data.drop(eliminate_col,axis=1,inplace=True)

?? ??? ??

In [ ]:
# temp_data['payment_plan_start_date']=pd.to_datetime(temp_data['payment_plan_start_date'], format='%b-%Y').dt.to_period('M')
temp_data['issue_d']=pd.to_datetime(temp_data['issue_d'], format='%b-%Y').dt.to_period('D')
temp_data['earliest_cr_line']=pd.to_datetime(temp_data['earliest_cr_line'], format='%b-%Y').dt.to_period('D')

# ?? ?? NA ? ??? ??? ??? ? ???? ???.

??

In [ ]:
idx_na_mths_since_recent_bc=temp_data[temp_data['mths_since_recent_bc'].isnull()].index
mths_since_recent_bc_median=temp_data['mths_since_recent_bc'].median()
temp_data.loc[idx_na_mths_since_recent_bc,'mths_since_recent_bc']=mths_since_recent_bc_median
idx_na_mths_since_recent_bc=temp_data[temp_data['mths_since_recent_inq'].isnull()].index
mths_since_recent_bc_median=temp_data['mths_since_recent_inq'].median()
temp_data.loc[idx_na_mths_since_recent_bc,'mths_since_recent_inq']=mths_since_recent_bc_median
temp_data['revol_util']=temp_data['revol_util'].str.replace('%', '').astype(float)
eliminate_list.append('dti')

??

In [ ]:
##### ?? NAN ?? ??? ???~
# last_pymn_d : ?? ??? ?? ???? ?? ???? => NAN ?? ??
# 1. No Payment ? '2000-01' ??? ??? ??? ???
temp_data['last_pymnt_d'].fillna(pd.to_datetime('2000-01', format='%Y-%m').to_period('M'), inplace=True)
# 2. issue_d ? ?? ????? (?? ?? ??? ??. ????? )( X ?? ?? ?? ??? ??)
## ??? ???? ?? d-day ? ?? ?? ? ??? ???? ? =>


##### ??
# charge_off = 0 : ?? 2007 ?? issue : 12 ?? ?? ????, 0 ?? ?????
temp_data['chargeoff_within_12_mths'].fillna(0, inplace=True)

#1. bc_util
#1.1 : ?? ?? total_rev_hi_lim ? nan ?(?? drop????) ??
#1.2 : ???? ??? ?? ?? -> ?? ???? ??? ??
#1.3 : ???? mean ?? ??? ???????, mean ?? ??
mean_bc_util = temp_data['bc_util'].mean()
temp_data['bc_util'].fillna(mean_bc_util, inplace=True)

#### ?? x

###2. il_util
#### => ?? ?? ~
## 2.1 total_bal_il (?? ?? ? . ?( ??? ) ??? -> 0 ?? ??, ??? 0 ?? )
## 2.2 ?? ???? ?? ?? ?? -> ??/ ?? ??? ??
## 2.3 ? ????, ??? 0 ??? ? 7??, ??? ??..
#total_bal_il ?? ???? 519721 ? ??, 0?? ?????? ==> ?? ?? ?????
# ???? ?? nan ?? 75%? or ....
numerator = temp_data['total_bal_il']
denominator = temp_data['total_il_high_credit_limit']
temp_data['il_util'] = temp_data['il_util'].fillna(numerator / denominator)


??

In [ ]:
# last_credit_pull_d : na ?? ??, ?? ??? ?? ????? ??

# mo_sin_old_il_acct : ????? ??
idx_na_mo_sin_old_il_acct = temp_data[temp_data['mo_sin_old_il_acct'].isnull()].index
mo_sin_old_il_acct_median = temp_data['mo_sin_old_il_acct'].median()
temp_data.loc[idx_na_mo_sin_old_il_acct, 'mo_sin_old_il_acct'] = mo_sin_old_il_acct_median

# pct_tl_nvr_dlq : ??????
idx_na_pct_tl_nvr_dlq = temp_data[temp_data['pct_tl_nvr_dlq'].isnull()].index  # ???? ?? ???? ??
pct_tl_nvr_dlq_non_na = temp_data['pct_tl_nvr_dlq'].dropna()  # ???? ?? ?? ??

np.random.seed(42)  # ?? ??
pct_tl_nvr_dlq_bootstrap_samples = np.random.choice(pct_tl_nvr_dlq_non_na, size=len(idx_na_pct_tl_nvr_dlq), replace=True)  # ???????? ?? ??

temp_data.loc[idx_na_pct_tl_nvr_dlq, 'pct_tl_nvr_dlq'] = pct_tl_nvr_dlq_bootstrap_samples  # ??? ??? ??? ??


# pub_rec_bankruptcies: linear regression (pub_rec? ????)

??

In [ ]:
mths1_med=temp_data['mths_since_recent_bc_dlq'].median()
mths2_med=temp_data['mths_since_recent_revol_delinq'].median()
temp_data['mths_since_recent_bc_dlq'].fillna(mths1_med, inplace=True)
temp_data['mths_since_recent_revol_delinq'].fillna(mths2_med, inplace=True)
replace_zero_list.append('revol_util')
replace_zero_list.append('bc_open_to_buy')
eliminate_col.append('num_sats')

??

In [ ]:
# ? 5?, ???

# mths_since_last_delinq : ??? ?(922,199)
## ???? ??? ?? ?? ??? ?? ?, ??? ?? ??? ????, ??? ?? ??? ?? ??? 0?? ??.
## ???? ???? loan_status ???, loan_status ?? ?? ?? ???? ??.
replace_zero_list.append('mths_since_last_delinq')
# mths_since_last_major_derog : ??? ?(1,321,878)
## ??? ?? ??? ? ??? ?? ?, ???? ?? ??? ???? ???? ????? ????, ?? 0?? ??
## ???? ???? loan_status ???, loan_status ?? ?? ?? ???? ??.
replace_zero_list.append('mths_since_last_major_derog')
# num_bc_sats # ?? ???? ?? ????? ? : (35,113)
## num_bc_sats ?? ???? ???? ??? ??? ??(Fully Paid)? ??? ??. ???, ???? ??? ???? ??? ???? ?? ??? ???? ??.
## ???? ? 4.85, ???? 4, ???? 77 ??(???) ????, ????? ??? ????? ????? ??
median_value = temp_data['num_bc_sats'].median()
temp_data['num_bc_sats'].fillna(median_value, inplace=True)

# num_tl_120dpd_2m # ?? 2?? ?? 120? ?? ??? ??? ? : (96,751)
## Fully Paid: 63,734?? ?? ??? ??? ?? ??(???? ?? ??? ? ???? Fully Paid ??? ????, ???? ??? ???? ??? ???? ??? ??? ?? ??)
## ??? ???? ?? 0?? ??? ?, ???? 0?? ??? ?? ????.
replace_zero_list.append('num_tl_120dpd_2m')

# tax_liens # ?? ???? ? : (73?)
## ???? ???, loan_status? Does not meet the credit policy? ??? ??, ? ? ???? Fully Paid ???.
## tax_liens ??? ???? ?? ?? ???? ??? ??? ???? ???? ??.
## ???? ?? 0?? tax_liens ?? 25%, 50%, 75% ??? ?? 0???, ???? 0?? ??? ?? ???? ??? ??? ???? ??.
replace_zero_list.append('tax_liens')

??

* mths_since_last_record : ?? ?? (?? ??? ?? ??)
* collections_12_mths_ex_med : ? ??
* percent_bc_gt_75: ????? ???, ????? ???? ???
* total_bal_ex_mort: ????? ???, ????? ???? ???
* total_bc_limit : ????? ???, ????? ???? ???
* mort_acc : ????? ???, ????? ???? ???

In [ ]:

# 1. null ?? ? ??
eliminate_list.append('collections_12_mths_ex_med')

# 2. ????? ???? ?? ?? ?? ??
def bootstrap_imputation(df, column, groupby_column):
    # ???? ???? ??
    for group in df[groupby_column].unique():
        group_data = df[df[groupby_column] == group]
        non_null_values = group_data[column].dropna().values

        if len(non_null_values) > 0:
            # ???? ?? ????? ???? ??
            sampled_values = np.random.choice(non_null_values, size=group_data[column].isna().sum(), replace=True)
            df.loc[(df[groupby_column] == group) & (df[column].isna()), column] = sampled_values

    return df

# 3. percent_bc_gt_75 ??? ??
temp_data = bootstrap_imputation(temp_data, 'percent_bc_gt_75', 'sub_grade')

# 4. total_bal_ex_mort ??? ??
temp_data = bootstrap_imputation(temp_data, 'total_bal_ex_mort', 'sub_grade')

# 5. total_bc_limit ??? ??
temp_data = bootstrap_imputation(temp_data, 'total_bc_limit', 'sub_grade')

# 6. mort_acc ??? ??
temp_data = bootstrap_imputation(temp_data, 'mort_acc', 'sub_grade')



In [ ]:
temp_data.loc[(temp_data['num_il_tl'].isna() |
               temp_data['total_bal_il'].isna() |
               temp_data['all_util'].isna()) &
    temp_data['il_util'].isna(),'il_util']=0

In [ ]:
temp_data.loc[(temp_data['num_il_tl'].isna() |
               temp_data['total_bal_il'].isna() |
               temp_data['all_util'].isna()) &
    temp_data['il_util'].isna(),'il_util'].head()

In [ ]:
temp_data = bootstrap_imputation(temp_data, 'il_util', 'sub_grade')

# ?? ??

In [ ]:
eliminate_index=[]
for i in eliminate_list:
    eliminate_index.append(temp_data[temp_data[i].isnull()].index.astype(int))
eliminate_index= list(set(item for sublist in eliminate_index for item in sublist))
temp_data.drop(eliminate_index,axis=0,inplace=True)

In [ ]:
for i in replace_zero_list:
    temp_data[i].fillna(0, inplace=True)

In [ ]:
temp_data.dropna(inplace=True)

In [ ]:
temp_data.isnull().sum()

In [ ]:
fico_zero_data = temp_data[temp_data['last_fico_range_high'] == 0]

for sub_grade in fico_zero_data['sub_grade'].unique():
    sub_grade_data = temp_data[(temp_data['sub_grade'] == sub_grade) & (temp_data['last_fico_range_high'] > 0)]

    if not sub_grade_data.empty:
        sampled_values = np.random.choice(sub_grade_data['last_fico_range_high'], size=len(fico_zero_data[fico_zero_data['sub_grade'] == sub_grade]), replace=True)
        temp_data.loc[(temp_data['sub_grade'] == sub_grade) & (temp_data['last_fico_range_high'] == 0), 'last_fico_range_high'] = sampled_values

temp_data['last_fico_range_high'].value_counts()
temp_data.drop(columns=['last_fico_range_low'], inplace=True)


idx_num_bc_tl = temp_data.loc[temp_data['num_bc_tl'] > 61].index
temp_data = temp_data.drop(idx_num_bc_tl)
idx_annual_inc = temp_data.loc[temp_data['annual_inc'] > 9999999.00].index
temp_data_filtered = temp_data.drop(idx_annual_inc)

In [ ]:
temp_data.shape

In [ ]:
cu_in_data=temp_data.loc[temp_data['loan_status']=='Current',:]
cu_out_data=temp_data.loc[temp_data['loan_status']!='Current',:]

In [ ]:
cu_in_data.to_csv('/Users/kbh/Downloads/lending club/all_clean_current_final_in_data.csv',index=False)
cu_out_data.to_csv('/Users/kbh/Downloads/lending club/all_clean_current_final_out_data.csv',index=False)

In [ ]:
temp_current=pd.read_csv('/Users/kbh/Downloads/lending club/all_clean_current_final_in_data.csv')
temp_out=pd.read_csv('/Users/kbh/Downloads/lending club/all_clean_current_final_out_data.csv')

In [ ]:
temp_out.head()